In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import sklearn.metrics as metrics
from scipy.signal import butter, sosfilt

DATA_PATH = csv_path = os.path.join("..", "data", "EMG-data.csv")

df = pd.read_csv(DATA_PATH)

SUBJECT_ROWS = [99980, 103636, 111012, 105175, 100859, 99670, 102134, 100225]
TOTAL_ROWS = sum(SUBJECT_ROWS)
CHANNELS = ["channel1", "channel2", "channel3", "channel4"]
CHANNELS_FILTERED = [c + '_filtered' for c in CHANNELS]
CHANNELS_NORMALIZED = [c + '_normalized' for c in CHANNELS]
SUBJECT_SEVEN_START = sum(SUBJECT_ROWS[0:6])
LAST_TWO_NUM_ROWS = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]

LOW_PASS_FREQ = 500
HIGH_PASS_FREQ = 30
SAMPLE_RATE = 1000

FEATURE_COLS = ["RMS", "waveform_len", "MAV", "max_abs", "min_abs", "std", "zero_crossings"]
FEATURE_COLS_MC = [f"{col}_{channel}" for channel in CHANNELS for col in FEATURE_COLS]

822691


In [2]:
def butterworth_filter(data, order=3, cutoff=HIGH_PASS_FREQ, fs=SAMPLE_RATE, filter_type='highpass'):
    if filter_type not in ['lowpass', 'highpass', 'bandpass', 'bandstop']:
        raise ValueError("Invalid filter type requested")
    sos = butter(N=order, Wn=cutoff, fs=SAMPLE_RATE, btype=filter_type, output='sos')
    filtered_data = sosfilt(sos, data)
    return filtered_data

def filter_emg_data(df, lpf=LOW_PASS_FREQ, hpf=HIGH_PASS_FREQ, fs=SAMPLE_RATE, order=3):
    df[CHANNELS_FILTERED] = df[CHANNELS].apply(lambda x: butterworth_filter(x, order=order, cutoff=hpf, fs=fs, filter_type='highpass'))
    #df[CHANNELS_FILTERED] = df[CHANNELS_FILTERED].apply(lambda x: butterworth_filter(x, order=order, cutoff=lpf, fs=fs, filter_type='lowpass'))
    return df

def normalize_window(window):
    window_mean = window.mean()
    window_std = window.std()
    current_val = window.iloc[-1]
    return (current_val - window_mean) / window_std
    
def snw_emg_data(df, window_size=200, filtered=True):
    if filtered:
        df[CHANNELS_NORMALIZED] = df[CHANNELS_FILTERED].rolling(window=window_size).apply(lambda x: normalize_window(x))
    else:
        df[CHANNELS_NORMALIZED] = df[CHANNELS].rolling(window=window_size).apply(lambda x: normalize_window(x))
    df['windowed_class'] = df['class'].rolling(window=window_size).apply(lambda x: x.iloc[0])
    df['windowed_subject'] = df['subject'].rolling(window=window_size).apply(lambda x: x.iloc[0])
    return df

def pre_process_emg(csv_path=DATA_PATH, lpf=LOW_PASS_FREQ, hpf=HIGH_PASS_FREQ, butter_order=3, 
                    normalize=False, norm_window_size=200):
    df = pd.read_csv(csv_path)
    df = filter_emg_data(df, lpf=lpf, hpf=hpf, order=butter_order)
    if normalize:
        df = snw_emg_data(df, window_size=norm_window_size, filtered=True)
    pre_processed_df = pd.DataFrame()
    if normalize:
        pre_processed_df[CHANNELS] = df[CHANNELS_NORMALIZED]
        pre_processed_df['class'] = df['windowed_class']
        pre_processed_df['subject'] = df['windowed_subject']
        return pre_processed_df
    else:
        pre_processed_df[CHANNELS] = df[CHANNELS_FILTERED]
        pre_processed_df['class'] = df['class']
        pre_processed_df['subject'] = df['subject']
        return pre_processed_df
        

def get_feature_df(df, read_channel='channel3', start_row=0, num_read=SUBJECT_ROWS[0]):
    """Given a path to EMG df with channel, gesture, class, subject cols, returns feature windowed data from specified 
    range, exlcuding specified channels omits windows in which all rows do not have the same class"""
    df = df.iloc[start_row : start_row + num_read].copy()
    dropped_channels = []
    for i in range(len(CHANNELS)):
        if read_channel != CHANNELS[i]:
            dropped_channels.append(CHANNELS[i])
    df = df.drop(columns=dropped_channels)

    windowed_class = df["class"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_class = windowed_class 
    windowed_subject = df["subject"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_rms_col = df[read_channel].pow(2).rolling(window=200, step=100).mean().pow(0.5)
    windowed_wfl_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(np.diff(x)).sum())
    windowed_stdev_col = df[read_channel].rolling(window=200, step=100).std()
    windowed_mav_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).mean())
    windowed_min_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).min())
    def crossings(s): 
        return (s.shift(1) * s < 0)
    windowed_zc_col = crossings(df[read_channel]).rolling(window=200, step=100).sum()
    windowed_max_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).max())
    
    rolling_class = df["class"].rolling(window=200, step=100)
    windowed_mask_col = rolling_class.min() == rolling_class.max()
    feature_df = pd.DataFrame({"RMS": windowed_rms_col,
                               "waveform_len": windowed_wfl_col,
                               "MAV": windowed_mav_col,
                               "max_abs": windowed_max_col,
                               "min_abs": windowed_min_col,
                               "std": windowed_stdev_col,
                               "zero_crossings": windowed_zc_col,
                               'mask': windowed_mask_col,
                               "class": windowed_class,
                               'subject': windowed_subject})
    feature_df = feature_df[feature_df['mask']]
    return feature_df

def get_feature_df_mc(df, start_row=0, num_read=SUBJECT_ROWS[0]):
    mc_df = None
    for channel in CHANNELS:
        df_ch = get_feature_df(df, read_channel=channel, start_row=start_row, num_read=num_read)
        channel_features = df_ch[FEATURE_COLS].add_suffix(f"_{channel}")
        if mc_df is None:
            mc_df = channel_features
            mc_df["class"] = df_ch["class"]
            mc_df["subject"] = df_ch["subject"]
        else:
            mc_df = pd.concat([mc_df, channel_features], axis=1)
    return mc_df

def get_train_test_df(whole_df, read_channel='channel3', train_start=0, train_read=sum(SUBJECT_ROWS[0:5]),
                  test_start=SUBJECT_SEVEN_START, test_read=LAST_TWO_NUM_ROWS):
    if (train_start + train_read > TOTAL_ROWS):
        raise ValueError('Requested train rows read out of bounds')
    if (test_start + test_read > TOTAL_ROWS):
        raise ValueError('Requested test rows read out of bounds')
    if (train_start > test_start and train_start < test_start + test_read):
        raise ValueError('Leakage in train and test sets')
    if (train_start + train_read > test_start and train_start + train_read < test_start + test_read):
            raise ValueError('Leakage in train and test sets')

    train_df = get_feature_df(whole_df,read_channel=read_channel, start_row=train_start, num_read=train_read)
    test_df = get_feature_df(whole_df, read_channel=read_channel, start_row=test_start, num_read=test_read)
    return train_df, test_df

In [ ]:
def get_df_features_labels(df, feature_cols=FEATURE_COLS):
    features = df[feature_cols].to_numpy()
    labels = df["class"].to_numpy()
    return features, labels

def evaluate_model(model, df, feature_cols=FEATURE_COLS):
    features, labels = get_df_features_labels(df, feature_cols)
    preds = model.predict(features)
    score = model.score(features, labels)
    return preds, score

def train_log_reg(df, feature_cols=FEATURE_COLS):
    features, labels = get_df_features_labels(df, feature_cols)
    reg = LogisticRegression(max_iter=10000)
    reg.fit(features, labels)
    return reg

def train_lda(df, feature_cols=FEATURE_COLS):
    features, labels = get_df_features_labels(df, feature_cols)
    lda = LinearDiscriminantAnalysis()
    lda.fit(features, labels)
    return lda

def evaluate_model_detailed(model, df, feature_cols=FEATURE_COLS, f1_average='macro'):
    preds, score = evaluate_model(model, df, feature_cols=feature_cols)
    labels = df['class'].to_numpy()

    f1 = metrics.f1_score(labels, preds, average=f1_average)
    bacc = metrics.balanced_accuracy_score(labels, preds)
    confusion_matrix = metrics.confusion_matrix(labels, preds)

    results = {
        'preds': preds,
        'score': score,
        'f1': f1,
        'bacc': bacc,
        'confusion matrix': confusion_matrix
    }
    return results
    

In [12]:
whole_df = pre_process_emg()
train_df, test_df = get_train_test_df(whole_df=whole_df)
reg = train_log_reg(train_df)

train_results = evaluate_model_detailed(reg, train_df)
test_results = evaluate_model_detailed(reg, test_df)

print('Single channel: ')
print(f'Train results:')
print(f"score: {train_results['score']}")
print(f"f1: {train_results['f1']}")
print(f"bacc: {train_results['bacc']}")
print(f"confusion matrix: {train_results['confusion matrix']}")

print(f'Test results: ')
print(f"score: {test_results['score']}")
print(f"f1: {test_results['f1']}")
print(f"bacc: {test_results['bacc']}")
print(f"confusion matrix: {test_results['confusion matrix']}")

Single channel: 
Train results:
score: 0.7248400232693426
f1: 0.7235444386066172
bacc: 0.7250859848578795
confusion matrix: [[1024    0    0    0    0]
 [   0  776    5  221   17]
 [   1    0  693   10  312]
 [   1  125   34  805  100]
 [   0    1  442  150  440]]
Test results: 
score: 0.6856287425149701
f1: 0.65208951675081
bacc: 0.6864321735600134
confusion matrix: [[ 59   0 343   0   0]
 [  0 326   8  73   7]
 [  0   0 371   0  27]
 [  0  59   0 330   2]
 [  0   0   0 111 288]]


In [5]:
train_df_mc = get_feature_df_mc(whole_df, start_row=0, num_read=sum(SUBJECT_ROWS[0:6]))
test_df_mc = get_feature_df_mc(whole_df, start_row=SUBJECT_SEVEN_START, num_read=LAST_TWO_NUM_ROWS)

reg_mc = train_log_reg(train_df_mc, feature_cols=FEATURE_COLS_MC)

train_results_mc = evaluate_model_detailed(reg_mc, train_df_mc, feature_cols=FEATURE_COLS_MC)
test_results_mc = evaluate_model_detailed(reg_mc, test_df_mc, feature_cols=FEATURE_COLS_MC)

print('Multi-channel:')
print('Train results:')
print(f"score: {train_results_mc['score']}")
print(f"f1: {train_results_mc['f1']}")
print(f"bacc: {train_results_mc['bacc']}")
print(f"confusion matrix:\n{train_results_mc['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_mc['score']}")
print(f"f1: {test_results_mc['f1']}")
print(f"bacc: {test_results_mc['bacc']}")
print(f"confusion matrix:\n{test_results_mc['confusion matrix']}")

Multi-channel:
Train results:
score: 0.94091796875
f1: 0.9414900502887326
bacc: 0.9418104754684891
confusion matrix:
[[1221    0    0    0    0]
 [   0 1182    5   29    1]
 [   0    2 1204    1    4]
 [   0   16   18 1055  174]
 [   0    3    2  108 1119]]
Test results:
score: 0.8782435129740519
f1: 0.8774070025471481
bacc: 0.8763447107317489
confusion matrix:
[[402   0   0   0   0]
 [  0 407   7   0   0]
 [  0   0 333   0  65]
 [  0  23   0 255 113]
 [  0   0   0  36 363]]


In [6]:
AMPLITUDE_FEATURE_COLS = ["RMS", "MAV", "max_abs", "min_abs", "std", "waveform_len"]
AMPLITUDE_FEATURE_COLS_MC = [f"{col}_{channel}" for channel in CHANNELS for col in AMPLITUDE_FEATURE_COLS]


def get_subject_rest_stats(feature_df, amplitude_cols, rest_class=0, max_samples=5000):
    """Per-subject mean and std of each amplitude feature during rest (class 0), simulating a short calibration period."""
    rest_df = feature_df[feature_df['class'] == rest_class].head(max_samples)
    rest_mean = rest_df.groupby('subject')[amplitude_cols].mean()
    rest_std = rest_df.groupby('subject')[amplitude_cols].std()
    return rest_mean, rest_std

def apply_baseline_calibration(feature_df, amplitude_cols, rest_class=0, max_samples=5000, eps=1e-8):
    """Z-score each subject's amplitude features against that subject's own rest distribution."""
    rest_mean, rest_std = get_subject_rest_stats(feature_df, amplitude_cols, rest_class=rest_class, max_samples=max_samples)
    calibrated_df = feature_df.copy()
    for subject in rest_mean.index:
        mask = calibrated_df['subject'] == subject
        calibrated_df.loc[mask, amplitude_cols] = (
            (calibrated_df.loc[mask, amplitude_cols] - rest_mean.loc[subject])
            / (rest_std.loc[subject] + eps)
        )
    return calibrated_df

In [7]:
train_df_cal = apply_baseline_calibration(train_df, AMPLITUDE_FEATURE_COLS)
test_df_cal = apply_baseline_calibration(test_df, AMPLITUDE_FEATURE_COLS)

reg_cal = train_log_reg(train_df_cal)

train_results_cal = evaluate_model_detailed(reg_cal, train_df_cal)
test_results_cal = evaluate_model_detailed(reg_cal, test_df_cal)

print('Single channel (baseline-calibrated):')
print('Train results:')
print(f"score: {train_results_cal['score']}")
print(f"f1: {train_results_cal['f1']}")
print(f"bacc: {train_results_cal['bacc']}")
print(f"confusion matrix:\n{train_results_cal['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_cal['score']}")
print(f"f1: {test_results_cal['f1']}")
print(f"bacc: {test_results_cal['bacc']}")
print(f"confusion matrix:\n{test_results_cal['confusion matrix']}")

Single channel (baseline-calibrated):
Train results:
score: 0.7515997673065736
f1: 0.75247056561664
bacc: 0.7515814120554114
confusion matrix:
[[1024    0    0    0    0]
 [   0  754    4  237   24]
 [   0    0  671   24  321]
 [   0  148   33  821   63]
 [   0    0  344   83  606]]
Test results:
score: 0.46407185628742514
f1: 0.41432616487626184
bacc: 0.46323879029742443
confusion matrix:
[[402   0   0   0   0]
 [  2  97  14  68 233]
 [129   0 269   0   0]
 [  0   1  79   0 311]
 [  0   0 237   0 162]]


In [8]:
train_df_mc_cal = apply_baseline_calibration(train_df_mc, AMPLITUDE_FEATURE_COLS_MC)
test_df_mc_cal = apply_baseline_calibration(test_df_mc, AMPLITUDE_FEATURE_COLS_MC)

reg_mc_cal = train_log_reg(train_df_mc_cal, feature_cols=FEATURE_COLS_MC)

train_results_mc_cal = evaluate_model_detailed(reg_mc_cal, train_df_mc_cal, feature_cols=FEATURE_COLS_MC)
test_results_mc_cal = evaluate_model_detailed(reg_mc_cal, test_df_mc_cal, feature_cols=FEATURE_COLS_MC)

print('Multi-channel (baseline-calibrated):')
print('Train results:')
print(f"score: {train_results_mc_cal['score']}")
print(f"f1: {train_results_mc_cal['f1']}")
print(f"bacc: {train_results_mc_cal['bacc']}")
print(f"confusion matrix:\n{train_results_mc_cal['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_mc_cal['score']}")
print(f"f1: {test_results_mc_cal['f1']}")
print(f"bacc: {test_results_mc_cal['bacc']}")
print(f"confusion matrix:\n{test_results_mc_cal['confusion matrix']}")

Multi-channel (baseline-calibrated):
Train results:
score: 0.96826171875
f1: 0.9686838646556467
bacc: 0.9686985551153697
confusion matrix:
[[1221    0    0    0    0]
 [   0 1206    4    6    1]
 [   0    0 1206    0    5]
 [   0    5    3 1165   90]
 [   0    1    1   79 1151]]
Test results:
score: 0.8862275449101796
f1: 0.8841246524232413
bacc: 0.8848583081680863
confusion matrix:
[[402   0   0   0   0]
 [  6 408   0   0   0]
 [  6   0 381   0  11]
 [  0  15   8 299  69]
 [  0   1   0 112 286]]


/Users/ckrenteras2024/Library/Python/3.11/lib/python/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
def get_subject_rest_stats_raw(df, channels=CHANNELS, rest_class=0, max_samples=None):
    """Per-subject mean and std of the raw (filtered) EMG channels during rest (class 0)."""
    rest_df = df[df['class'] == rest_class]
    if max_samples is not None:
        rest_df = rest_df.groupby('subject').head(max_samples)
    rest_mean = rest_df.groupby('subject')[channels].mean()
    rest_std = rest_df.groupby('subject')[channels].std()
    return rest_mean, rest_std

def apply_baseline_calibration_raw(df, channels=CHANNELS, rest_class=0, max_samples=None, eps=1e-8):
    """Z-score each subject's raw (filtered) EMG channels against that subject's own rest distribution,
    i.e. calibrate the signal itself, before any feature extraction."""
    rest_mean, rest_std = get_subject_rest_stats_raw(df, channels=channels, rest_class=rest_class, max_samples=max_samples)
    calibrated_df = df.copy()
    for subject in rest_mean.index:
        mask = calibrated_df['subject'] == subject
        calibrated_df.loc[mask, channels] = (
            (calibrated_df.loc[mask, channels] - rest_mean.loc[subject])
            / (rest_std.loc[subject] + eps)
        )
    return calibrated_df

def get_feature_df_mc_calibrated_raw(df, start_row=0, num_read=SUBJECT_ROWS[0], rest_class=0, max_samples=None):
    """Baseline-calibrate the raw EMG signal per subject, then run the standard multi-channel feature
    extraction on the calibrated signal (normalize-then-extract, as opposed to extract-then-normalize)."""
    calibrated_df = apply_baseline_calibration_raw(df, channels=CHANNELS, rest_class=rest_class, max_samples=max_samples)
    return get_feature_df_mc(calibrated_df, start_row=start_row, num_read=num_read)

In [10]:
train_df_mc_raw_cal = get_feature_df_mc_calibrated_raw(whole_df, start_row=0, num_read=sum(SUBJECT_ROWS[0:6]))
test_df_mc_raw_cal = get_feature_df_mc_calibrated_raw(whole_df, start_row=SUBJECT_SEVEN_START, num_read=LAST_TWO_NUM_ROWS)

reg_mc_raw_cal = train_log_reg(train_df_mc_raw_cal, feature_cols=FEATURE_COLS_MC)

train_results_mc_raw_cal = evaluate_model_detailed(reg_mc_raw_cal, train_df_mc_raw_cal, feature_cols=FEATURE_COLS_MC)
test_results_mc_raw_cal = evaluate_model_detailed(reg_mc_raw_cal, test_df_mc_raw_cal, feature_cols=FEATURE_COLS_MC)

print('Multi-channel (raw EMG baseline-calibrated before feature extraction):')
print('Train results:')
print(f"score: {train_results_mc_raw_cal['score']}")
print(f"f1: {train_results_mc_raw_cal['f1']}")
print(f"bacc: {train_results_mc_raw_cal['bacc']}")
print(f"confusion matrix:\n{train_results_mc_raw_cal['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_mc_raw_cal['score']}")
print(f"f1: {test_results_mc_raw_cal['f1']}")
print(f"bacc: {test_results_mc_raw_cal['bacc']}")
print(f"confusion matrix:\n{test_results_mc_raw_cal['confusion matrix']}")

Multi-channel (raw EMG baseline-calibrated before feature extraction):
Train results:
score: 0.92236328125
f1: 0.9230792264309879
bacc: 0.9234203151829325
confusion matrix:
[[1221    0    0    0    0]
 [   0 1172   12   15   18]
 [   0   15 1176    1   19]
 [   0   21   11 1001  230]
 [   0   13   11  111 1097]]
Test results:
score: 0.6497005988023952
f1: 0.6118752789859621
bacc: 0.6465624739061515
confusion matrix:
[[402   0   0   0   0]
 [100 299   2   1  12]
 [202   0 190   0   6]
 [ 43   0  40  61 247]
 [  4   0  42   3 350]]


/Users/ckrenteras2024/Library/Python/3.11/lib/python/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
whole_df_raw_cal = apply_baseline_calibration_raw(whole_df)
train_df_cal_raw, test_df_cal_raw = get_train_test_df(whole_df=whole_df_raw_cal)

reg_cal_raw = train_log_reg(train_df_cal_raw)

train_results_cal_raw = evaluate_model_detailed(reg_cal_raw, train_df_cal_raw)
test_results_cal_raw = evaluate_model_detailed(reg_cal_raw, test_df_cal_raw)

print('Single channel (raw EMG baseline-calibrated before feature extraction):')
print('Train results:')
print(f"score: {train_results_cal_raw['score']}")
print(f"f1: {train_results_cal_raw['f1']}")
print(f"bacc: {train_results_cal_raw['bacc']}")
print(f"confusion matrix:\n{train_results_cal_raw['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_cal_raw['score']}")
print(f"f1: {test_results_cal_raw['f1']}")
print(f"bacc: {test_results_cal_raw['bacc']}")
print(f"confusion matrix:\n{test_results_cal_raw['confusion matrix']}")

Single channel (raw EMG baseline-calibrated before feature extraction):
Train results:
score: 0.7673065735892961
f1: 0.7680009607645557
bacc: 0.7675950042277899
confusion matrix:
[[1024    0    0    0    0]
 [   0  722    4  252   41]
 [   0    3  751   34  228]
 [   0  171   33  794   67]
 [   0    4  309   54  666]]
Test results:
score: 0.312874251497006
f1: 0.239724276703469
bacc: 0.3125021241473066
confusion matrix:
[[402   0   0   0   0]
 [  4  29 106  93 182]
 [202   0 196   0   0]
 [  0   0 344   0  47]
 [  0   0 399   0   0]]


/Users/ckrenteras2024/Library/Python/3.11/lib/python/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
SNW_WINDOW_SIZE = 500  # longer than the default 200-sample window

whole_df_snw = pre_process_emg(normalize=True, norm_window_size=SNW_WINDOW_SIZE)

train_df_snw, test_df_snw = get_train_test_df(whole_df=whole_df_snw)

reg_snw = train_log_reg(train_df_snw)

train_results_snw = evaluate_model_detailed(reg_snw, train_df_snw)
test_results_snw = evaluate_model_detailed(reg_snw, test_df_snw)

print(f'Single channel (sliding-window normalized, window={SNW_WINDOW_SIZE}):')
print('Train results:')
print(f"score: {train_results_snw['score']}")
print(f"f1: {train_results_snw['f1']}")
print(f"bacc: {train_results_snw['bacc']}")
print(f"confusion matrix:\n{train_results_snw['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_snw['score']}")
print(f"f1: {test_results_snw['f1']}")
print(f"bacc: {test_results_snw['bacc']}")
print(f"confusion matrix:\n{test_results_snw['confusion matrix']}")

train_df_mc_snw = get_feature_df_mc(whole_df_snw, start_row=0, num_read=sum(SUBJECT_ROWS[0:6]))
test_df_mc_snw = get_feature_df_mc(whole_df_snw, start_row=SUBJECT_SEVEN_START, num_read=LAST_TWO_NUM_ROWS)

reg_mc_snw = train_log_reg(train_df_mc_snw, feature_cols=FEATURE_COLS_MC)

train_results_mc_snw = evaluate_model_detailed(reg_mc_snw, train_df_mc_snw, feature_cols=FEATURE_COLS_MC)
test_results_mc_snw = evaluate_model_detailed(reg_mc_snw, test_df_mc_snw, feature_cols=FEATURE_COLS_MC)

print(f'\nMulti-channel (sliding-window normalized, window={SNW_WINDOW_SIZE}):')
print('Train results:')
print(f"score: {train_results_mc_snw['score']}")
print(f"f1: {train_results_mc_snw['f1']}")
print(f"bacc: {train_results_mc_snw['bacc']}")
print(f"confusion matrix:\n{train_results_mc_snw['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_mc_snw['score']}")
print(f"f1: {test_results_mc_snw['f1']}")
print(f"bacc: {test_results_mc_snw['bacc']}")
print(f"confusion matrix:\n{test_results_mc_snw['confusion matrix']}")

Single channel (sliding-window normalized, window=500):
Train results:
score: 0.4732142857142857
f1: 0.46621894849477163
bacc: 0.4727124904804539
confusion matrix:
[[854   8  44 116   2]
 [ 54 391 152 257 165]
 [ 30 182 385 185 234]
 [110 191 130 564  70]
 [ 12 223 337 212 244]]
Test results:
score: 0.3821178821178821
f1: 0.34429399964673624
bacc: 0.38315441878307754
confusion matrix:
[[  5   4 393   0   0]
 [ 27 157  57 111  62]
 [  3   3 318  12  62]
 [ 26  39 116 187  23]
 [  3  59 185  52  98]]



Multi-channel (sliding-window normalized, window=500):
Train results:
score: 0.6753542922300049
f1: 0.6709640926951123
bacc: 0.6744905120669984
confusion matrix:
[[966  82  73  52  48]
 [ 88 903 143  19  64]
 [131 204 577  86 213]
 [ 34  17  52 975 185]
 [ 34 110 144 214 725]]
Test results:
score: 0.4070929070929071
f1: 0.36192860601879956
bacc: 0.40580808697991505
confusion matrix:
[[ 14 196   7   0 185]
 [ 41 290  51   5  27]
 [  7 128  60  13 190]
 [ 29   1  23 180 158]
 [  9  12  35  70 271]]


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
